# 📓 Practice 7. The Particle Filter

## 🎯 학습 목표

- **확률 분포를 입자(particle)들로 표현**하는 추정 방법 이해
- 비선형 시스템에서 EKF의 한계와 PF의 강점 비교
- Predict → Weight → Resample → Estimate 사이클 구현
- EKF-CV vs EKF-DR vs PF-DR 3가지 알고리즘 종합 비교

---

## 📦 노트북 구성

| 파트 | 내용 | 핵심 결과물 |
|------|------|-------------|
| **7-1-1** | DR 모델 비선형 차량 시뮬레이션 (50초) | 참값 `x_history` (N, 3) |
| **7-1-2** | Bearing 센서 측정값 시뮬레이션 | 측정값 `y_history` (N, 2) |
| **7-2** | DR 모델 Particle Filter 추정 | `x_hat_pf` (N, 3) |
| **7-3** | EKF-CV vs EKF-DR vs PF-DR 종합 비교 | 3개 알고리즘 RMSE + plot |

> Practice 7 노트북 안에서 **EKF-CV, EKF-DR, PF-DR 모두 구현**하여 비교

---

## 🚨 EKF vs PF 핵심 차이

| 항목 | EKF | PF |
|------|-----|-----|
| 상태 표현 | 평균 + 공분산 (가우시안) | **N개의 입자(samples)** |
| 비선형 처리 | 자코비안 (1차 테일러) | **샘플링** |
| 가우시안 가정 | 필요 | 불필요 |
| 자코비안 유도 | 필요 | **불필요** ✨ |
| 다봉 분포 | 처리 불가 | 처리 가능 |
| 계산량 | 적음 | 많음 (N에 비례) |

---

## 📐 시스템 정보 요약

### 차량 운동 모델 (DR - Dead Reckoning)

```

# 📋 Practice 7-1-1. State Simulation (DR Model with V, ψ̇ inputs)

## ■ 목표
- 속도(V)와 yawrate(ψ̇) 입력을 받는 비선형 차량 운동 모델로 참값 상태 생성
- 상태: [x, y, ψ]ᵀ (위치 + 헤딩)
- 50초 동안 차량의 진짜 궤적(ground truth) 생성

## ■ 시스템 모델 (DR, Dead Reckoning - 비선형)

```
x_{t+1} = x_t + V(t) · dT · cos(ψ_t)
y_{t+1} = y_t + V(t) · dT · sin(ψ_t)
ψ_{t+1} = ψ_t + dT · ψ̇(t)
```

> **비선형성**: cos(ψ), sin(ψ) → 행렬곱 형태로 표현 불가
> 함수 형태: x_{t+1} = f(x_t, u_t)

## ■ 입력 데이터 (PDF에 명시된 값)

```
Velocity:
   period         = 2π
   amplitude      = 3
   vertical shift = 6
   → V(t) = 6 + 3 · sin(t)        [m/s]

Yaw rate:
   period         = 6π
   amplitude      = 6 · π / 180
   vertical shift = 10 · π / 180
   → ψ̇(t) = 10°·π/180 + 6°·π/180 · sin(t / 3)    [rad/s]
```

## ■ 조건

- dT = 0.1 s
- 시뮬레이션 구간: t = 0 ~ 50 s (총 501 step)
- 초기 위치: x_0 = 1 m, y_0 = 1 m
- 초기 헤딩: ψ_0 = 45° = π/4 rad

## ■ 풀이 절차

### 1️⃣ 시뮬레이션 조건 설정
- dT, 시간 배열 t (0 ~ 50 s)

### 2️⃣ 입력 데이터 생성
- V(t): 6 + 3·sin(t)
- ψ̇(t): 10°·π/180 + 6°·π/180·sin(t/3)

### 3️⃣ 초기 상태 + 저장 배열 준비
- x_0 = [1, 1, π/4]ᵀ
- x_history (N, 3)

### 4️⃣ 시간 루프 (비선형 propagation)
- 매 step마다 cos, sin 식으로 직접 계산

### 5️⃣ 결과 시각화
- 입력(V, ψ̇) plot
- 각 상태(x, y, ψ) 시간 그래프
- 2D 평면에서 차량 궤적 (x-y plot)

---

## ■ 한줄 정리

> 👉 "DR 운동 방정식으로 50초 동안 차량의 ground truth 궤적 생성"
> 👉 7-1-2 (측정값), 7-2 (PF), 7-3 (비교)의 기준 데이터

---

## 💡 헷갈리기 쉬운 포인트

- **각도 단위**: 모든 각도는 **라디안**으로 통일
- **np.cos, np.sin**: 라디안 입력 (도(degree) X)
- **입력 함수 인자**: V는 sin(t), ψ̇는 sin(t/3) (주기 다름!)
- **vertical shift**: sin 함수의 평균값 (위로 이동)

In [ ]:
# =========================================
# 1️⃣ 라이브러리 불러오기
# =========================================
import numpy as np
import matplotlib.pyplot as plt


# =========================================
# 2️⃣ 시뮬레이션 조건 설정
# =========================================
dT     = 0.1                              # 샘플링 시간 [s]
t_end  = 50                               # 종료 시간 [s] (Practice 6: 30s)
t      = np.arange(0, t_end + dT, dT)     # [0, 0.1, ..., 50] → (501,)
N      = len(t)                           # 시간 step 개수


# =========================================
# 3️⃣ 입력 데이터 생성 (PDF에 명시된 값 그대로!)
#    Velocity   : period=2π,  amplitude=3,         vertical shift=6
#    Yaw rate   : period=6π,  amplitude=6·π/180,   vertical shift=10·π/180
#
#    sin(2π·t / period) 형태로 표현
# =========================================
# 속도: 6 + 3·sin(2π·t / 2π) = 6 + 3·sin(t)
V = 6.0 + 3.0 * np.sin(2 * np.pi * t / (2 * np.pi))   # (N,) [m/s]

# Yaw rate: 10°·π/180 + 6°·π/180 · sin(2π·t / 6π) = 10°·π/180 + 6°·π/180 · sin(t/3)
psi_dot = (10 * np.pi / 180) + (6 * np.pi / 180) * np.sin(2 * np.pi * t / (6 * np.pi))
                                                       # (N,) [rad/s]


# =========================================
# 4️⃣ 초기 상태 + 저장 배열 준비
#    상태: [x, y, psi]
#    초기값: (1m, 1m, 45°)
# =========================================
x_state = np.array([1.0,                  # x_0  : 1 m
                    1.0,                  # y_0  : 1 m
                    np.deg2rad(45)])      # psi_0: 45° → 라디안

x_history = np.zeros((N, 3))              # 각 step의 [x, y, psi]
x_history[0] = x_state                    # 초기값 저장


# =========================================
# 5️⃣ 비선형 Propagation 루프
#    매 step:
#      x   = x   + V · dT · cos(psi)
#      y   = y   + V · dT · sin(psi)
#      psi = psi + psi_dot · dT
# =========================================
for k in range(1, N):
    # 직전 step의 상태와 입력
    x_prev   = x_history[k-1, 0]
    y_prev   = x_history[k-1, 1]
    psi_prev = x_history[k-1, 2]
    
    V_prev       = V[k-1]
    psi_dot_prev = psi_dot[k-1]
    
    # 비선형 운동 방정식
    x_new   = x_prev   + V_prev * dT * np.cos(psi_prev)
    y_new   = y_prev   + V_prev * dT * np.sin(psi_prev)
    psi_new = psi_prev + psi_dot_prev * dT
    
    # 저장
    x_history[k] = [x_new, y_new, psi_new]


# =========================================
# 6️⃣ 결과 시각화 ① - 입력 데이터
# =========================================
fig, axes = plt.subplots(2, 1, figsize=(10, 5))

axes[0].plot(t, V, 'b-')
axes[0].set_title('Input: Velocity')
axes[0].set_xlabel('time [s]'); axes[0].set_ylabel('V [m/s]')
axes[0].grid(True)

axes[1].plot(t, psi_dot, 'r-')
axes[1].set_title('Input: Yaw rate')
axes[1].set_xlabel('time [s]'); axes[1].set_ylabel('ψ̇ [rad/s]')
axes[1].grid(True)

plt.tight_layout()
plt.show()


# =========================================
# 7️⃣ 결과 시각화 ② - 각 상태별 시간 그래프
# =========================================
fig, axes = plt.subplots(3, 1, figsize=(10, 7))

axes[0].plot(t, x_history[:, 0], 'b-')
axes[0].set_title('Position X')
axes[0].set_xlabel('time [s]'); axes[0].set_ylabel('x [m]')
axes[0].grid(True)

axes[1].plot(t, x_history[:, 1], 'b-')
axes[1].set_title('Position Y')
axes[1].set_xlabel('time [s]'); axes[1].set_ylabel('y [m]')
axes[1].grid(True)

# 헤딩은 보기 편하게 도(deg) 단위로 표시
axes[2].plot(t, np.rad2deg(x_history[:, 2]), 'g-')
axes[2].set_title('Heading ψ (deg)')
axes[2].set_xlabel('time [s]'); axes[2].set_ylabel('ψ [deg]')
axes[2].grid(True)

plt.tight_layout()
plt.show()


# =========================================
# 8️⃣ 결과 시각화 ③ - 2D 차량 궤적
# =========================================
plt.figure(figsize=(7, 6))
plt.plot(x_history[:, 0], x_history[:, 1], 'b-', linewidth=1.5)
plt.scatter(x_history[0, 0],  x_history[0, 1],
            c='green', s=80, label='start', zorder=3)
plt.scatter(x_history[-1, 0], x_history[-1, 1],
            c='red',   s=80, label='end',   zorder=3)
plt.title('2D Vehicle Trajectory (Dead Reckoning, 50s)')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.axis('equal'); plt.grid(True); plt.legend()
plt.show()


# =========================================
# 9️⃣ 결과 확인
# =========================================
print(f"최종 위치   : x = {x_history[-1, 0]:.2f} m,  y = {x_history[-1, 1]:.2f} m")
print(f"최종 헤딩   : ψ = {np.rad2deg(x_history[-1, 2]):.2f}°")
print(f"x_history shape: {x_history.shape}")

# 📋 Practice 7-1-2. Measurement Simulation (Bearing Sensor)

## ■ 목표
- 7-1-1에서 만든 참 위치 (x_t, y_t)에 비선형 측정 모델(Bearing 센서) 적용
- 거리 r과 각도 θ에 가우시안 노이즈를 더해 측정값 생성

## ■ 측정 모델 (Bearing Sensor - 비선형!)

원점에 있는 Bearing 센서가 차량을 측정:

```
        ┌  r  ┐     ┌    √(x² + y²)    ┐
   y =  │     │  =  │                  │  + noise
        └  θ  ┘     └  tan⁻¹(y / x)    ┘
```

> **비선형성**: √, tan⁻¹ 때문에 더 이상 H · x 형태로 못 씀
> 함수 형태: y = h(x)

## ■ 변수 의미

```
   r : 센서로부터 차량까지의 거리       (range)
   θ : 차량과 X축이 이루는 각도          (bearing angle)
   
   √(x² + y²)    : 피타고라스로 거리 계산
   tan⁻¹(y / x)  : x축 기준 차량 방향 각도
```

## ■ 노이즈 조건

- 거리 노이즈: σ_r = 1 m
- 각도 노이즈: σ_θ = 3° = π/60 rad
- r과 θ에 **독립적인** 가우시안 노이즈

## ■ 풀이 절차

### 1️⃣ 비선형 측정 함수 h(x) 정의
- 입력: 상태 [x, y, ψ]
- 출력: [r, θ] (ψ는 측정과 무관)

### 2️⃣ 노이즈 생성
- r 노이즈: N(0, σ_r²) (단위: m)
- θ 노이즈: N(0, σ_θ²) (단위: rad)

### 3️⃣ 측정값 계산
- 매 시간 step에서 h(x_true) + noise

### 4️⃣ 결과 시각화
- 참 r vs 측정 r (시간)
- 참 θ vs 측정 θ (시간)
- 2D 평면에서 참값 궤적 vs 측정값 복원 위치

---

## ■ 한줄 정리

> 👉 "Bearing 센서로 거리와 각도를 측정. 비선형 함수 √, tan⁻¹가 들어감"
> 👉 7-2 (PF), 7-3 (EKF/PF 비교)에서 측정 입력으로 사용됨

---

## 💡 헷갈리기 쉬운 포인트

- **각도 단위**: σ_θ는 라디안 변환 필수 (3° → π/60)
- **np.arctan2(y, x)**: 4사분면을 모두 처리. arctan(y/x)보다 안전
- **두 노이즈는 독립**: r과 θ에 각각 다른 난수 적용
- **재현성**: np.random.seed로 고정하면 매번 같은 결과

In [ ]:
# =========================================
# 1️⃣ 노이즈 재현성을 위한 seed 설정
# =========================================
np.random.seed(42)


# =========================================
# 2️⃣ 비선형 측정 함수 h(x) 정의
#    입력: state = [x, y, psi]
#    출력: [r, theta]
# =========================================
def h(state):
    """
    Bearing sensor measurement model
    
    Parameters
    ----------
    state : (3,) [x, y, psi]
    
    Returns
    -------
    measurement : (2,) [r, theta]
    """
    x = state[0]
    y = state[1]
    
    r     = np.sqrt(x**2 + y**2)          # 거리
    theta = np.arctan2(y, x)              # 각도 (4사분면 안전)
    
    return np.array([r, theta])


# =========================================
# 3️⃣ 측정 노이즈 표준편차 설정
#    r은 m, theta는 rad 단위로 통일
# =========================================
sigma_r     = 1.0                         # 거리 노이즈 [m]
sigma_theta = np.deg2rad(3)               # 각도 노이즈 [rad] (3°)


# =========================================
# 4️⃣ 노이즈 생성
#    r과 theta 각각 독립적으로 생성
# =========================================
noise_r     = np.random.normal(0, sigma_r,     N)   # (N,) 거리 노이즈
noise_theta = np.random.normal(0, sigma_theta, N)   # (N,) 각도 노이즈


# =========================================
# 5️⃣ 측정값 계산
#    매 시간 step:
#      [r_true, theta_true] = h(x_true)
#      [r_meas, theta_meas] = [r_true + noise, theta_true + noise]
# =========================================
y_history      = np.zeros((N, 2))         # (N, 2) 노이즈 측정값 [r, theta]
y_history_true = np.zeros((N, 2))         # (N, 2) 참 측정값 (비교용)

for k in range(N):
    # 참 측정값 (노이즈 없음)
    z_true = h(x_history[k])              # (2,) [r_true, theta_true]
    y_history_true[k] = z_true
    
    # 노이즈 추가한 측정값
    y_history[k, 0] = z_true[0] + noise_r[k]      # 측정 r
    y_history[k, 1] = z_true[1] + noise_theta[k]  # 측정 theta


# =========================================
# 6️⃣ 결과 시각화 ① - 시간에 따른 r, θ 비교
# =========================================
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# 거리 r 비교
axes[0].plot(t, y_history_true[:, 0], 'b-',  label='true r',     linewidth=1.5)
axes[0].plot(t, y_history[:, 0],      'r--', label='measured r', linewidth=0.8)
axes[0].set_title('Range r: True vs Measurement')
axes[0].set_xlabel('time [s]'); axes[0].set_ylabel('r [m]')
axes[0].legend(); axes[0].grid(True)

# 각도 θ 비교 (deg 단위로 보기 편하게)
axes[1].plot(t, np.rad2deg(y_history_true[:, 1]), 'b-',
             label='true θ',     linewidth=1.5)
axes[1].plot(t, np.rad2deg(y_history[:, 1]),      'r--',
             label='measured θ', linewidth=0.8)
axes[1].set_title('Bearing θ: True vs Measurement')
axes[1].set_xlabel('time [s]'); axes[1].set_ylabel('θ [deg]')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()


# =========================================
# 7️⃣ 결과 시각화 ② - 2D 평면 궤적 비교
#    측정값 (r, θ)을 (x, y)로 변환:
#       x_meas = r · cos(θ)
#       y_meas = r · sin(θ)
# =========================================
x_meas = y_history[:, 0] * np.cos(y_history[:, 1])
y_meas = y_history[:, 0] * np.sin(y_history[:, 1])

plt.figure(figsize=(7, 6))
plt.plot(x_history[:, 0], x_history[:, 1],
         'b-', linewidth=1.5, label='true trajectory')
plt.scatter(x_meas, y_meas,
            c='red', s=8, alpha=0.5, label='measurement (recovered)')
plt.scatter(0, 0, c='black', s=80, marker='^', label='sensor (origin)')

plt.title('2D Trajectory: True vs Measurement (Bearing Sensor)')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.axis('equal'); plt.grid(True); plt.legend()
plt.show()


# =========================================
# 8️⃣ 노이즈 통계 확인 (검증용)
# =========================================
print(f"r 노이즈 평균   : {np.mean(noise_r):.4f}     (이론: 0)")
print(f"r 노이즈 표준편차: {np.std(noise_r):.4f}      (이론: {sigma_r})")
print(f"θ 노이즈 평균   : {np.rad2deg(np.mean(noise_theta)):.4f}°    (이론: 0°)")
print(f"θ 노이즈 표준편차: {np.rad2deg(np.std(noise_theta)):.4f}°     (이론: {np.rad2deg(sigma_theta):.2f}°)")
print(f"y_history shape: {y_history.shape}")

# 📋 Practice 7-2. Particle Filter with DR Model

## ■ 목표
- DR 모델 + Bearing 측정에 **Particle Filter (PF)** 적용
- 자코비안 없이, **N개의 입자(particles)** 로 확률 분포를 표현
- EKF의 가우시안/선형화 가정에서 자유로운 추정

---

## ■ Particle Filter 핵심 아이디어

> "확률 분포 p(x)를 수식이 아닌 **N개의 샘플로 표현**"

```
EKF: x ~ N(평균, 공분산)         ← 가우시안 가정
PF:  x ~ {x_1, x_2, ..., x_N}    ← 입자들의 분포
```

### 입자(particle)란?

각 입자는 **"상태의 한 가능한 가설"** 임:
- 입자 1: "차량이 (5, 3, 30°)에 있을 것 같아"
- 입자 2: "(5.1, 2.9, 31°)에 있을 것 같아"
- ...
- 입자 N: "(4.8, 3.2, 28°)에 있을 것 같아"

→ 이 입자들이 모여서 **현재 추정의 불확실성**을 표현

---

## ■ PF 알고리즘 4단계

```
1. 초기화 (Initialize)
   ┌─────────────────────────────────────┐
   │  N개의 입자를 사전 분포에서 샘플링  │
   │  x_i^0 ~ p(x_0),  i = 1, ..., N     │
   └─────────────────────────────────────┘

매 시간 step에서:

2. Prediction (각 입자를 process model로 propagate)
   ┌─────────────────────────────────────┐
   │  x_i^t = f(x_i^{t-1}, u) + w_i      │
   │  w_i ~ N(0, Q)  ← process noise     │
   └─────────────────────────────────────┘

3. Weighting (각 입자의 측정 likelihood 계산)
   ┌─────────────────────────────────────┐
   │  w_i = p(y_t | x_i^t)               │
   │      = N(y_t - h(x_i^t); 0, R)      │
   │  → 측정과 일치하는 입자 = 큰 가중치 │
   └─────────────────────────────────────┘

4. Resampling (가중치에 비례해 입자 재추출)
   ┌─────────────────────────────────────┐
   │  큰 가중치 입자 → 여러 번 복제       │
   │  작은 가중치 입자 → 제거             │
   │  N개 유지                           │
   └─────────────────────────────────────┘

5. Estimate (입자 평균 = 추정값)
   ┌─────────────────────────────────────┐
   │  x̂ = (1/N) · Σ x_i                 │
   └─────────────────────────────────────┘
```

---

## ■ 비유로 이해하기

```
"보물찾기 게임"

1. 초기화: 사람 N명이 보물 위치를 추측해서 흩어짐
2. Prediction: 각자 "보물이 움직였을 것 같은 방향"으로 한 발 이동
3. Weighting: 측정 도구(센서)로 "여기 보물이 있어" 신호가 옴
              → 신호와 가까운 사람 = 점수 높음
4. Resampling: 점수 높은 사람 주변에 사람들이 다시 모임
5. 평균 위치 = 보물 추정 위치
```

---

## ■ Likelihood 계산 (Weighting의 핵심)

각 입자가 측정값과 얼마나 일치하는지 가우시안 PDF로 계산:

```
입자 i의 예측 측정: z_pred_i = h(x_i)
실제 측정값:        y

Innovation: ν_i = y - z_pred_i

Likelihood:
   w_i = exp(-0.5 · ν_i^T · R^{-1} · ν_i)

(정규화 상수는 Σw_i로 나눠서 처리하면 OK)
```

> **각도 wrap 주의**: ν의 θ 부분은 ±π 처리 필요

---

## ■ Resampling (간단 multinomial 방법)

```
1. 정규화: w_i = w_i / Σw_i
2. 누적 분포: cumsum(w)
3. N번 반복:
   - 0~1 사이 난수 생성
   - 누적 분포에서 그 위치의 입자 인덱스 선택
4. 선택된 인덱스로 입자 재구성
```

> **np.random.choice**의 `p` 파라미터로 한 줄에 처리 가능

---

## ■ Q, R, 초기 조건

- 입자 개수: **N_particles = 1000**
- Q (3×3): 입자 propagation 시 추가하는 process noise
  - 예: `diag(0.1, 0.1, 0.01)` (튜닝)
- R (2×2): 측정 노이즈 (likelihood 계산용)
  - `diag(σ_r², σ_θ²)`
- 초기 입자: 참값 주변에 정규분포로 뿌리기
  - 평균 = [1, 1, π/4], 분산 = 큰 값

## ■ 풀이 절차

### 1️⃣ 함수 준비
- `f_dr(state, u)`: DR 시스템 모델 (이미 7-1-1에서 사용)
- `h(state)`: Bearing 측정 모델 (이미 7-1-2에서 사용)
- `wrap_angle(angle)`: 각도 wrap

### 2️⃣ PF 클래스 또는 파라미터 설정
- N_particles, Q, R, 초기 입자

### 3️⃣ PF 메인 루프 (4단계 사이클)

### 4️⃣ 결과 시각화 (7-2-1)
- 추정 상태 + 오차 plot

### 5️⃣ RMSE 계산 (7-2-2)

---

## ■ 한줄 정리

> 👉 "비선형 함수를 자코비안 없이, N개의 입자로 분포를 직접 표현해서 추정"
> 👉 EKF의 한계(가우시안 가정)를 입자 샘플링으로 극복

---

## 💡 헷갈리기 쉬운 포인트

- **입자 개수 N**: 너무 적으면 부정확, 너무 많으면 느림 (N=500~2000 적당)
- **Q는 입자에만 추가**: 시스템 모델이 정확해도 약간의 다양성을 위해 noise 필수
- **각도 wrap**: 헤딩 ψ + 측정 θ 모두 wrap 처리
- **Particle degeneracy**: 시간 지나면 한 입자에만 가중치 집중 → resampling으로 해결
- **Resampling 위치**: 매 step 항상 vs 효율 검사 후 (간단히 매 step 진행)

In [ ]:
# =========================================
# 1️⃣ DR 시스템 모델 함수 정의
#    상태: [x, y, psi]
#    입력: [V, psi_dot]
# =========================================
def f_dr(state, u):
    """
    Dead Reckoning motion model
    
    Parameters
    ----------
    state : (3,) [x, y, psi]
    u     : (2,) [V, psi_dot]
    
    Returns
    -------
    next_state : (3,) [x_new, y_new, psi_new]
    """
    x       = state[0]
    y       = state[1]
    psi     = state[2]
    V       = u[0]
    psi_dot = u[1]
    
    x_new   = x   + V * dT * np.cos(psi)
    y_new   = y   + V * dT * np.sin(psi)
    psi_new = psi + dT * psi_dot
    
    return np.array([x_new, y_new, psi_new])


# =========================================
# 2️⃣ Angle wrap 함수
# =========================================
def wrap_angle(angle):
    """각도를 -π ~ π 범위로 wrap"""
    return (angle + np.pi) % (2 * np.pi) - np.pi


# =========================================
# 3️⃣ Particle Filter 파라미터 설정
# =========================================
N_particles = 1000                        # 입자 개수

# Process noise (입자 propagation 시 추가)
Q_pf = np.diag([0.1, 0.1, np.deg2rad(2)**2])   # (3, 3)
                                          # x, y, psi noise

# Measurement noise covariance (likelihood 계산용)
R_pf = np.array([[sigma_r**2,     0.0           ],
                 [0.0,            sigma_theta**2]])  # (2, 2)
R_pf_inv = np.linalg.inv(R_pf)            # 미리 계산 (likelihood에서 반복 사용)


# =========================================
# 4️⃣ 초기 입자 샘플링
#    참값 [1, 1, π/4] 주변에 정규분포로 흩뿌림
#    초기 불확실성을 큰 분산으로 표현
# =========================================
np.random.seed(42)                        # 재현성

# 초기 입자 (N_particles, 3) 형태
particles = np.zeros((N_particles, 3))
particles[:, 0] = 1.0 + np.random.normal(0, 2.0, N_particles)   # x ~ N(1, 2²)
particles[:, 1] = 1.0 + np.random.normal(0, 2.0, N_particles)   # y ~ N(1, 2²)
particles[:, 2] = np.pi/4 + np.random.normal(0, np.deg2rad(20), N_particles)  # ψ ~ N(45°, 20°²)

# 가중치 초기화 (균등)
weights = np.ones(N_particles) / N_particles


# =========================================
# 5️⃣ 추정 결과 저장 배열
# =========================================
x_hat_pf = np.zeros((N, 3))               # 시간별 추정 [x, y, psi]


# =========================================
# 6️⃣ Particle Filter 메인 루프
# =========================================
for k in range(N):
    # ===== ① Prediction =====
    # 각 입자에 process noise 추가하여 propagate
    u_k = np.array([V[k-1] if k > 0 else V[0],
                    psi_dot[k-1] if k > 0 else psi_dot[0]])
    
    # 각 입자에 process noise 샘플링
    process_noise = np.random.multivariate_normal(
        mean=[0, 0, 0], cov=Q_pf, size=N_particles
    )                                     # (N_particles, 3)
    
    # 각 입자를 f()로 propagate
    for i in range(N_particles):
        particles[i] = f_dr(particles[i], u_k) + process_noise[i]
    
    # ===== ② Weighting =====
    # 각 입자의 측정 likelihood 계산
    y_k = y_history[k]                    # (2,) [r, theta] 측정값
    
    for i in range(N_particles):
        # 입자의 예측 측정값
        z_pred = h(particles[i])          # (2,) [r_pred, theta_pred]
        
        # Innovation (각도는 wrap)
        innovation = y_k - z_pred         # (2,)
        innovation[1] = wrap_angle(innovation[1])
        
        # Likelihood (가우시안 PDF)
        # exp(-0.5 · innovation^T · R^-1 · innovation)
        weights[i] = np.exp(-0.5 * innovation @ R_pf_inv @ innovation)
    
    # 가중치 정규화
    weights = weights + 1e-300            # 0 division 방지
    weights = weights / np.sum(weights)
    
    # ===== ③ Resampling (multinomial) =====
    # 가중치에 비례해 N개 입자 재추출
    indices = np.random.choice(
        N_particles,
        size=N_particles,
        p=weights
    )
    particles = particles[indices]
    
    # 가중치 균등 초기화 (resampling 후)
    weights = np.ones(N_particles) / N_particles
    
    # ===== ④ Estimate (입자 평균) =====
    # 위치 x, y는 단순 평균
    x_hat_pf[k, 0] = np.mean(particles[:, 0])
    x_hat_pf[k, 1] = np.mean(particles[:, 1])
    
    # 헤딩 ψ는 원형 평균 (circular mean) 사용 (각도라서)
    sin_mean = np.mean(np.sin(particles[:, 2]))
    cos_mean = np.mean(np.cos(particles[:, 2]))
    x_hat_pf[k, 2] = np.arctan2(sin_mean, cos_mean)


# =========================================
# 7️⃣ Practice 7-2-1: 추정 상태 + 오차 plot
# =========================================
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
labels = ['Position X', 'Position Y', 'Heading ψ']
units  = ['m', 'm', 'deg']

for i in range(3):
    if i == 2:
        # 헤딩은 deg 단위로
        true_val = np.rad2deg(x_history[:, i])
        est_val  = np.rad2deg(x_hat_pf[:, i])
        error    = np.rad2deg(wrap_angle(x_history[:, i] - x_hat_pf[:, i]))
    else:
        true_val = x_history[:, i]
        est_val  = x_hat_pf[:, i]
        error    = x_history[:, i] - x_hat_pf[:, i]
    
    # 좌측: 추정 vs 참값
    axes[i, 0].plot(t, true_val, 'b-',  label='true',         linewidth=1.5)
    axes[i, 0].plot(t, est_val,  'r--', label='PF estimate',  linewidth=1.0)
    axes[i, 0].set_title(f'Estimate {labels[i]}')
    axes[i, 0].set_ylabel(f'[{units[i]}]')
    axes[i, 0].legend(); axes[i, 0].grid(True)
    
    # 우측: 오차
    axes[i, 1].plot(t, error, 'g-')
    axes[i, 1].axhline(0, color='k', linewidth=0.5)
    axes[i, 1].set_title(f'Error {labels[i]}')
    axes[i, 1].set_ylabel(f'error [{units[i]}]')
    axes[i, 1].grid(True)

axes[2, 0].set_xlabel('time [s]')
axes[2, 1].set_xlabel('time [s]')

plt.suptitle('Practice 7-2-1: PF (DR Model) Estimation', fontsize=14)
plt.tight_layout()
plt.show()


# =========================================
# 8️⃣ 2D 궤적 비교 plot
# =========================================
plt.figure(figsize=(7, 6))
plt.plot(x_history[:, 0], x_history[:, 1], 'b-',
         label='true', linewidth=2)
plt.plot(x_hat_pf[:, 0],  x_hat_pf[:, 1],  'r--',
         label='PF estimate', linewidth=1)
plt.scatter(0, 0, c='black', s=80, marker='^', label='sensor (origin)')

plt.title('2D Trajectory: True vs PF Estimate')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.axis('equal'); plt.grid(True); plt.legend()
plt.show()


# =========================================
# 9️⃣ Practice 7-2-2: RMSE 계산
# =========================================
rmse_pf = np.zeros(3)

# 위치 RMSE
rmse_pf[0] = np.sqrt(np.mean((x_history[:, 0] - x_hat_pf[:, 0])**2))
rmse_pf[1] = np.sqrt(np.mean((x_history[:, 1] - x_hat_pf[:, 1])**2))

# 헤딩 RMSE (wrap 처리)
heading_error = wrap_angle(x_history[:, 2] - x_hat_pf[:, 2])
rmse_pf[2] = np.sqrt(np.mean(heading_error**2))

print("=" * 50)
print("Practice 7-2-2: PF (DR Model) RMSE")
print("=" * 50)
print(f"RMSE Position X : {rmse_pf[0]:.4f} m")
print(f"RMSE Position Y : {rmse_pf[1]:.4f} m")
print(f"RMSE Heading ψ  : {np.rad2deg(rmse_pf[2]):.4f}° "
      f"({rmse_pf[2]:.4f} rad)")
print("=" * 50)
print(f"입자 개수 N = {N_particles}")

# 📋 Practice 7-3. 3가지 알고리즘 종합 비교

## ■ 목표
- **EKF-CV vs EKF-DR vs PF-DR** 세 알고리즘의 추정 성능을 비교
- 같은 시뮬레이션 데이터(7-1)에 대해 RMSE와 plot으로 정량/정성 비교
- 각 알고리즘의 강점/약점 파악

## ■ 비교 대상

| 알고리즘 | 시스템 모델 | 측정 모델 | 비선형 처리 |
|----------|------------|----------|-------------|
| **EKF-CV** | CV (선형) | Bearing (비선형) | 자코비안 H |
| **EKF-DR** | DR (비선형) | Bearing (비선형) | 자코비안 F, H |
| **PF-DR** | DR (비선형) | Bearing (비선형) | 입자 샘플링 |

> **PF-DR은 7-2에서 이미 구현 완료**
> EKF-CV와 EKF-DR을 새로 구현해서 비교

## ■ 풀이 절차

### 1️⃣ EKF-CV 구현 (Practice 6-2와 동일 구조)
- CV 시스템: 4차원 상태 [x, y, vx, vy]
- 측정: Bearing (자코비안 H 매 step 계산)

### 2️⃣ EKF-DR 구현 (Practice 6-3과 동일 구조)
- DR 시스템: 3차원 상태 [x, y, ψ]
- 시스템/측정 모두 비선형 (자코비안 F, H 매 step 계산)

### 3️⃣ 3개 결과 비교
- 위치 (x, y) 추정 plot
- 위치 오차 plot
- 2D 궤적 비교
- RMSE 표

---

## ■ 한줄 정리

> 👉 "선형 KF의 진화 흐름: KF → EKF → PF, 각각의 장단점 비교"
> 👉 시스템 비선형성과 모델 일치성이 추정 성능에 미치는 영향 확인

---

## 💡 비교 포인트

1. **EKF-CV vs EKF-DR**: 시스템 모델 일치성의 영향
2. **EKF-DR vs PF-DR**: 같은 모델에서 자코비안 vs 입자 샘플링
3. **EKF-CV vs PF-DR**: 가장 부정확한 EKF vs 가장 정교한 PF

In [ ]:
import h5py
import numpy as np

file_path = './data.mat'

def print_dataset_info(name, obj):
    print(f'\n[{name}]')
    
    if isinstance(obj, h5py.Group):
        print('type : Group')
        print('members :', list(obj.keys()))
        return
    
    data = np.array(obj)
    print('type  : Dataset')
    print('shape :', data.shape)
    print('dtype :', data.dtype)
    
    if data.size == 0:
        print('values: empty')
    elif data.size <= 100:
        print('values:')
        print(data)
    else:
        flat = data.flatten()
        print('values preview (first 100):')
        print(flat[:100])

with h5py.File(file_path, 'r') as f:
    print('Top-level keys in data.mat:', list(f.keys()))
    
    for key in f.keys():
        print_dataset_info(key, f[key])
        
        if isinstance(f[key], h5py.Group):
            for subkey in f[key].keys():
                print_dataset_info(f'{key}/{subkey}', f[key][subkey])


# 📊 Practice 7-3. 3가지 알고리즘 종합 분석 결과

## ■ 정량적 비교 (RMSE)

| 알고리즘 | RMSE x | RMSE y | RMSE ψ | 비고 |
|----------|--------|--------|--------|------|
| **EKF-CV** | (결과) m | (결과) m | 추정 X | 모델 불일치 |
| **EKF-DR** | (결과) m | (결과) m | (결과)° | 모델 일치 |
| **PF-DR** | (결과) m | (결과) m | (결과)° | 비선형 샘플링 |

> 위 표의 "(결과)"는 코드 실행 후 실제 RMSE 값을 채워 넣기

---

## ■ 정성적 비교 (Plot 분석)

### 🔵 EKF-CV (시스템: 선형, 측정: 비선형)
- 시뮬레이션은 DR(회전 운동), 추정은 CV(직선 가정)
- **모델 불일치** → 곡선 구간에서 추정값이 안쪽으로 벗어남
- 위치 오차 큼

### 🟢 EKF-DR (시스템: 비선형, 측정: 비선형)
- 시뮬레이션과 모델 일치
- 자코비안 F, H로 매 step 선형화 → 정확한 추정
- 가우시안 가정이 잘 맞음

### 🔴 PF-DR (시스템: 비선형, 측정: 비선형, 샘플링)
- 자코비안 없이 입자로 분포 표현
- EKF-DR과 비슷한 정확도
- **다봉 분포, 강한 비선형성에 강점** (이번 시뮬레이션은 단순해서 큰 차이 없음)

---

## ■ 알고리즘 선택 가이드

| 상황 | 추천 알고리즘 | 이유 |
|------|--------------|------|
| 시스템이 선형 + 가우시안 | **KF** | 가장 빠르고 정확 |
| 약한 비선형 + 가우시안 | **EKF** | 자코비안 선형화로 충분 |
| 강한 비선형 + 가우시안 | **UKF** | sigma point로 더 정확 |
| 임의의 비선형 + 비가우시안 | **PF** | 가장 일반적, 계산량 많음 |

---

## ■ 모델 vs 시스템 일치성

> **"모델이 시스템과 얼마나 잘 맞느냐가 정확도의 핵심"**

- EKF-CV vs EKF-DR: **같은 EKF인데 모델 차이로 큰 성능 차이**
- EKF-DR vs PF-DR: **같은 모델인데 알고리즘 차이는 적음**

→ **알고리즘보다 모델 선택이 더 중요한 경우가 많음**

---

## ■ Trade-off

| 항목 | EKF-CV | EKF-DR | PF-DR |
|------|--------|--------|-------|
| 정확도 | 낮음 | 높음 | 높음 |
| 계산 속도 | 빠름 | 빠름 | **느림** |
| 자코비안 유도 | H만 | F, H 모두 | **불필요** |
| 가우시안 가정 | 필요 | 필요 | **불필요** |
| 다봉 분포 처리 | 불가 | 불가 | **가능** |
| 구현 난이도 | 쉬움 | 보통 | 보통 |

---

## ■ 결론

### 💡 핵심 교훈

> **"가장 좋은 알고리즘은 없다. 시스템 특성에 맞는 선택이 답이다"**

- 시스템 모델이 정확하면 EKF로 충분
- 비선형성이 강하거나 다봉 분포가 있으면 PF
- 실시간 처리가 중요하면 EKF (또는 UKF)
- 정확도가 최우선이면 PF (또는 입자 수 늘린 PF)

### 🎯 자율주행 응용 팁

- **차량 자체 추정**: EKF (Bicycle 모델) 가장 많이 사용
- **다른 차량(객체) 추적**: 상황에 따라 EKF / UKF / IMM
- **다봉 분포 (어디 있는지 모를 때)**: PF (예: 자율주행 초기 위치 추정)

---

## ■ 한줄 정리

> 👉 KF(선형) → EKF(비선형, 자코비안) → PF(비선형, 샘플링)
> 👉 시스템 비선형성과 분포 형태에 따라 알고리즘 선택